# Trabalho 2 - Previsao de rendimento agricola

Tema: agronegocio. Dataset: Kaggle `patelris/crop-yield-prediction-dataset`. Estudo X: Yan et al. (2025).

Este notebook prepara o ambiente, baixa os dados, executa os modelos e mostra os resultados.

## 0. Antes de executar no Google Colab

Se for usar GPU, selecione no menu: `Runtime > Change runtime type > T4 GPU`.

Para baixar do Kaggle, voce pode usar uma destas opcoes:

1. Tentar o download anonimo com `kagglehub`, que costuma funcionar para datasets publicos.
2. Fazer upload do arquivo `kaggle.json`, gerado na sua conta Kaggle em `Account > API > Create New Token`.

Nao coloque `kaggle.json` no GitHub.

## 1. Obter o projeto

Se este notebook ja estiver dentro da pasta do projeto, execute apenas a segunda celula. Se estiver em um notebook vazio no Colab, preencha `GITHUB_REPO_URL` e execute a celula de clone.

In [ ]:
# Opcional: use esta celula se estiver abrindo um Colab vazio.
# Exemplo: GITHUB_REPO_URL = 'https://github.com/seu-usuario/trab2-am.git'
GITHUB_REPO_URL = ''

if GITHUB_REPO_URL:
    !git clone {GITHUB_REPO_URL} trab2-am
    %cd trab2-am
else:
    print('Pulando clone. Execute a proxima celula para conferir a pasta atual.')

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
print('Pasta atual:', root)
print('Arquivos:', sorted(p.name for p in root.iterdir())[:20])

## 2. Instalar dependencias

Use `requirements-colab.txt` para GPU e `requirements.txt` para CPU/local.

In [ ]:
USE_GPU = True

if USE_GPU:
    !pip install -q -r requirements-colab.txt
else:
    !pip install -q -r requirements.txt

## 3. Configurar Kaggle opcionalmente

Execute esta celula somente se o download anonimo falhar ou se o Kaggle pedir autenticacao. Ela abre um upload manual para o arquivo `kaggle.json`.

Como gerar o arquivo: Kaggle > foto do perfil > Account > API > Create New Token.

In [ ]:
# Opcional: autenticar Kaggle com upload de kaggle.json.
# Deixe comentado se o download anonimo funcionar.

# from google.colab import files
# uploaded = files.upload()
# if 'kaggle.json' in uploaded:
#     !mkdir -p ~/.kaggle
#     !cp kaggle.json ~/.kaggle/kaggle.json
#     !chmod 600 ~/.kaggle/kaggle.json
#     print('kaggle.json configurado com seguranca no ambiente temporario do Colab.')

## 4. Verificar GPU

In [ ]:
import torch

print('CUDA disponivel:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Sem GPU. Se quiser usar GPU, altere o tipo de runtime para T4 GPU.')

## 5. Baixar o dataset do Kaggle

In [ ]:
!python src/download_data.py

## 6. Executar experimento recomendado

Para testar rapido no Colab com GPU, use `--folds 3 --sample-size 6000`. Para resultado final, use `--folds 5`.

In [ ]:
RUN_FAST_TEST = True

if USE_GPU:
    if RUN_FAST_TEST:
        !python src/train_evaluate_gpu_colab.py --folds 3 --sample-size 6000
    else:
        !python src/train_evaluate_gpu_colab.py --folds 5
else:
    !python src/train_evaluate.py --sample-size 2500 --n-jobs 1

## 7. Rodar resultado final completo

Depois que o teste rapido funcionar, execute esta celula para gerar o resultado final GPU.

In [ ]:
# !python src/train_evaluate_gpu_colab.py --folds 5

## 8. Ver resultados

In [ ]:
import pandas as pd
from pathlib import Path

gpu_results = Path('outputs/results_gpu/model_results_gpu.csv')
cpu_results = Path('outputs/results/model_results.csv')

if gpu_results.exists():
    display(pd.read_csv(gpu_results))
elif cpu_results.exists():
    display(pd.read_csv(cpu_results))
else:
    print('Nenhum resultado encontrado. Execute a etapa de treinamento primeiro.')

## 9. Ver graficos

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figure_paths = [
    'outputs/figures_gpu/gpu_01_distribuicao_rendimento.png',
    'outputs/figures_gpu/gpu_02_matriz_correlacao.png',
    'outputs/figures_gpu/gpu_03_rendimento_por_cultura.png',
    'outputs/figures_gpu/gpu_04_comparacao_modelos.png',
    'outputs/figures_gpu/gpu_05_erros_modelos.png',
    'outputs/figures_gpu/gpu_06_r2_modelos.png',
    'outputs/figures_gpu/gpu_07_real_vs_predito_melhor_modelo.png',
    'outputs/figures_gpu/gpu_08_residuos_melhor_modelo.png',
    'outputs/figures_gpu/gpu_09_residuos_por_modelo.png',
    'outputs/figures_gpu/gpu_10_real_vs_predito_por_modelo.png',
    'outputs/figures_gpu/gpu_11_rendimento_medio_por_ano.png',
    'outputs/figures_gpu/gpu_12_chuva_vs_rendimento.png',
    'outputs/figures_gpu/gpu_13_temperatura_vs_rendimento.png',
    'outputs/figures/01_distribuicao_rendimento.png',
    'outputs/figures/02_matriz_correlacao.png',
    'outputs/figures/03_rendimento_por_cultura.png',
    'outputs/figures/04_comparacao_modelos.png',
    'outputs/figures/05_real_vs_predito_melhor_modelo.png',
    'outputs/figures/06_importancia_atributos.png',
]

for path in figure_paths:
    if Path(path).exists():
        print(path)
        display(Image(filename=path))

## 10. Baixar artefatos

Use esta celula no Colab para compactar resultados e graficos.

In [ ]:
# !zip -r resultados_trabalho2.zip outputs docs README.md requirements.txt requirements-colab.txt
# from google.colab import files
# files.download('resultados_trabalho2.zip')